# AprioriTID — Thực nghiệm và đánh giá (Chương 4)

Notebook này chạy tất cả thực nghiệm bắt buộc cho Chương 4 của báo cáo.

In [ ]:
# Load dependencies
using Plots
using Random
using Printf

Random.seed!(42)  # Reproducibility

# Load algorithm (in dependency order)
include(joinpath(@__DIR__, "..", "src", "structures.jl"))
include(joinpath(@__DIR__, "..", "src", "utils.jl"))
include(joinpath(@__DIR__, "..", "src", "algorithm", "apriori_gen.jl"))
include(joinpath(@__DIR__, "..", "src", "algorithm", "apriori_tid.jl"))

# Paths
const DATA_DIR = joinpath(@__DIR__, "..", "data")
const BENCHMARK_DIR = joinpath(DATA_DIR, "benchmark")
const TOY_DIR = joinpath(DATA_DIR, "toy")

# Helper
mean(x) = sum(x) / length(x)

# Plot settings
default(size=(800, 500), legend=:topright, linewidth=2, markersize=5)

println("Setup complete.")

## Load Benchmark Datasets

In [ ]:
# Load all benchmark datasets
datasets = Dict{String, Vector{Vector{Int}}}()
for name in ["chess", "mushrooms", "retail", "T10I4D100K"]
    fpath = joinpath(BENCHMARK_DIR, "$name.txt")
    if isfile(fpath)
        datasets[name] = read_spmf(fpath)
        n = length(datasets[name])
        avg_len = round(mean(length.(datasets[name])), digits=1)
        println("$name: $n transactions, avg length $avg_len")
    else
        println("WARNING: $name.txt not found")
    end
end

## Experiment (a): Kiểm tra tính đúng đắn (Correctness)

So sánh kết quả của AprioriTID với kết quả tham chiếu từ SPMF.

In [ ]:
# Correctness check on toy datasets (known expected results)
println("=== Correctness Check on Toy Datasets ===")
println()

# Example 1
trans1 = read_spmf(joinpath(TOY_DIR, "example1.txt"))
res1 = apriori_tid(trans1, 2)
expected1 = Set([
    ([1], 4), ([2], 3), ([4], 3), ([5], 3),
    ([1,2], 3), ([1,4], 3), ([1,5], 2), ([2,4], 2), ([4,5], 2),
    ([1,2,4], 2), ([1,4,5], 2)
])
match1 = Set([(sort(is), sup) for (is, sup) in res1]) == expected1
println("Example 1 (minsup=2): $(length(res1)) itemsets, match=$(match1) ✓")

# Example 2
trans2 = read_spmf(joinpath(TOY_DIR, "example2.txt"))
res2 = apriori_tid(trans2, 2)
println("Example 2 (minsup=2): $(length(res2)) itemsets (expected 87): $(length(res2)==87 ? "✓" : "✗")")

# Single item
trans3 = read_spmf(joinpath(TOY_DIR, "test_single_item.txt"))
res3 = apriori_tid(trans3, 2)
println("Single item (minsup=2): $(length(res3)) itemsets (expected 1): $(length(res3)==1 ? "✓" : "✗")")

# Empty result
trans4 = read_spmf(joinpath(TOY_DIR, "test_empty_result.txt"))
res4 = apriori_tid(trans4, 3)
println("Empty result (minsup=3): $(length(res4)) itemsets (expected 0): $(length(res4)==0 ? "✓" : "✗")")

# All frequent
trans5 = read_spmf(joinpath(TOY_DIR, "test_all_frequent.txt"))
res5 = apriori_tid(trans5, 2)
println("All frequent (minsup=2): $(length(res5)) itemsets (expected 7): $(length(res5)==7 ? "✓" : "✗")")

In [ ]:
# Correctness check on benchmark datasets
# Run at moderate minsup and report counts
println("=== Correctness Check on Benchmark Datasets ===")
println()

benchmark_configs = [
    ("chess", 2500),
    ("mushrooms", 4000),
    ("retail", 1000),
    ("T10I4D100K", 1000)
]

for (name, ms) in benchmark_configs
    if haskey(datasets, name)
        results = apriori_tid(datasets[name], ms)
        n_itemsets = length(results)
        max_k = maximum(length(is) for (is, _) in results)
        println("$name (minsup=$ms): $n_itemsets frequent itemsets, max length=$max_k")
    end
end

println()
println("Note: Compare these counts with SPMF output to verify correctness.")
println("To generate SPMF reference, run SPMF's AprioriTID with the same minsup values.")

## Experiment (b): Thời gian chạy theo minsup

In [ ]:
# Define minsup ranges for each dataset
minsup_configs = Dict(
    "chess" => [2800, 2600, 2400, 2200, 2000, 1800, 1600],
    "mushrooms" => [6000, 5000, 4000, 3000, 2500, 2000, 1500],
    "retail" => [1500, 1000, 750, 500, 300, 200, 100],
    "T10I4D100K" => [2000, 1500, 1000, 750, 500, 300, 200]
)

# Run experiments and collect timing data
timing_results = Dict{String, Vector{Tuple{Int, Float64, Int}}}()  # name -> [(minsup, time_ms, n_itemsets)]

for (name, minsups) in minsup_configs
    if !haskey(datasets, name)
        continue
    end
    trans = datasets[name]
    timing_results[name] = Tuple{Int, Float64, Int}[]
    
    # Warmup run
    apriori_tid(trans, minsups[1])
    
    for ms in minsups
        elapsed = @elapsed begin
            results = apriori_tid(trans, ms)
        end
        push!(timing_results[name], (ms, elapsed * 1000, length(results)))
        @printf("  %s minsup=%d: %.1f ms, %d itemsets\n", name, ms, elapsed*1000, length(results))
    end
end

In [ ]:
# Plot runtime vs minsup for each dataset
for (name, data) in timing_results
    minsups = [d[1] for d in data]
    times = [d[2] for d in data]
    
    p = plot(minsups, times,
        xlabel="minsup",
        ylabel="Thời gian (ms)",
        title="Thời gian chạy theo minsup — $name",
        marker=:circle,
        label="AprioriTID (Julia)",
        legend=:topleft
    )
    display(p)
    
    # Save to file
    savefig(p, joinpath(@__DIR__, "..", "docs", "runtime_$(name).pdf"))
end

## Experiment (c): Số lượng frequent itemset theo minsup

In [ ]:
# Plot number of frequent itemsets vs minsup
for (name, data) in timing_results
    minsups = [d[1] for d in data]
    counts = [d[3] for d in data]
    
    p = plot(minsups, counts,
        xlabel="minsup",
        ylabel="Số lượng frequent itemsets",
        title="Số lượng itemset theo minsup — $name",
        marker=:square,
        label="$name",
        legend=:topleft,
        yscale=:log10
    )
    display(p)
    
    savefig(p, joinpath(@__DIR__, "..", "docs", "itemset_count_$(name).pdf"))
end

## Experiment (d): Sử dụng bộ nhớ

In [ ]:
# Memory usage comparison: basic vs optimized
println("=== Peak Memory Usage (bytes allocated) ===")
println()

memory_configs = [
    ("chess", 2500),
    ("mushrooms", 4000),
    ("retail", 500),
    ("T10I4D100K", 1000)
]

mem_results = []

for (name, ms) in memory_configs
    if !haskey(datasets, name)
        continue
    end
    trans = datasets[name]
    
    # Warmup
    apriori_tid(trans, ms; optimized=true)
    apriori_tid(trans, ms; optimized=false)
    
    # Measure optimized
    mem_opt = @allocated apriori_tid(trans, ms; optimized=true)
    
    # Measure basic
    mem_basic = @allocated apriori_tid(trans, ms; optimized=false)
    
    push!(mem_results, (name, ms, mem_basic, mem_opt))
    @printf("  %s (minsup=%d): Basic=%.1f MB, Optimized=%.1f MB, Ratio=%.2f\n",
        name, ms, mem_basic/1e6, mem_opt/1e6, mem_opt/mem_basic)
end

In [ ]:
# Bar chart: memory comparison
if !isempty(mem_results)
    names = [r[1] for r in mem_results]
    mem_basic = [r[3] / 1e6 for r in mem_results]
    mem_opt = [r[4] / 1e6 for r in mem_results]
    
    p = groupedbar([mem_basic mem_opt],
        bar_position=:dodge,
        bar_width=0.35,
        xticks=(1:length(names), names),
        xlabel="Dataset",
        ylabel="Bộ nhớ (MB)",
        title="Sử dụng bộ nhớ: Basic vs Optimized",
        label=["Basic" "Optimized"],
        legend=:topright
    )
    display(p)
    savefig(p, joinpath(@__DIR__, "..", "docs", "memory_comparison.pdf"))
end

## Experiment (e): Khả năng mở rộng (Scalability)

In [ ]:
# Scalability test on Retail dataset
if haskey(datasets, "retail")
    retail = datasets["retail"]
    n = length(retail)
    
    fractions = [0.10, 0.25, 0.50, 0.75, 1.0]
    scale_ms = 500  # Fixed minsup
    
    scale_results = Tuple{Float64, Int, Float64}[]  # (fraction, n_trans, time_ms)
    
    # Warmup
    apriori_tid(retail[1:div(n,10)], scale_ms)
    
    for frac in fractions
        n_sub = round(Int, n * frac)
        subset = retail[1:n_sub]
        # Adjust minsup proportionally
        adj_ms = max(1, round(Int, scale_ms * frac))
        
        elapsed = @elapsed begin
            results = apriori_tid(subset, adj_ms)
        end
        push!(scale_results, (frac, n_sub, elapsed * 1000))
        @printf("  %.0f%% (%d trans, minsup=%d): %.1f ms, %d itemsets\n",
            frac*100, n_sub, adj_ms, elapsed*1000, length(results))
    end
    
    # Plot
    sizes = [r[2] for r in scale_results]
    times = [r[3] for r in scale_results]
    
    p = plot(sizes, times,
        xlabel="Số giao dịch",
        ylabel="Thời gian (ms)",
        title="Khả năng mở rộng — Retail",
        marker=:circle,
        label="AprioriTID",
        legend=:topleft
    )
    display(p)
    savefig(p, joinpath(@__DIR__, "..", "docs", "scalability_retail.pdf"))
end

## Experiment (f): Ảnh hưởng của độ dài giao dịch trung bình

In [ ]:
# Generate synthetic datasets with increasing average transaction length
Random.seed!(42)

n_items_pool = 50
n_transactions = 5000
avg_lengths = [5, 10, 15, 20, 25]
synth_ms = 100  # Fixed minsup

length_results = Tuple{Int, Float64, Int}[]  # (avg_len, time_ms, n_itemsets)

for avg_len in avg_lengths
    # Generate random transactions
    synth_trans = Vector{Int}[]
    for _ in 1:n_transactions
        len = max(1, round(Int, avg_len + randn() * 2))
        len = min(len, n_items_pool)
        items = sort!(randperm(n_items_pool)[1:len])
        push!(synth_trans, items)
    end
    
    # Warmup
    if avg_len == avg_lengths[1]
        apriori_tid(synth_trans, synth_ms)
    end
    
    elapsed = @elapsed begin
        results = apriori_tid(synth_trans, synth_ms)
    end
    push!(length_results, (avg_len, elapsed * 1000, length(results)))
    @printf("  avg_len=%d: %.1f ms, %d itemsets\n", avg_len, elapsed*1000, length(results))
end

# Plot
lens = [r[1] for r in length_results]
times = [r[2] for r in length_results]

p = plot(lens, times,
    xlabel="Độ dài giao dịch trung bình",
    ylabel="Thời gian (ms)",
    title="Ảnh hưởng của độ dài giao dịch\n(5000 trans, 50 items, minsup=100)",
    marker=:diamond,
    label="AprioriTID",
    legend=:topleft
)
display(p)
savefig(p, joinpath(@__DIR__, "..", "docs", "transaction_length_effect.pdf"))

## Experiment: So sánh Basic vs Optimized (Runtime)

In [ ]:
# Compare basic vs optimized runtime
println("=== Basic vs Optimized Runtime Comparison ===")
println()

opt_configs = [
    ("chess", 2500),
    ("mushrooms", 4000),
    ("retail", 500),
    ("T10I4D100K", 1000)
]

opt_results = []

for (name, ms) in opt_configs
    if !haskey(datasets, name)
        continue
    end
    trans = datasets[name]
    
    # Warmup both
    apriori_tid(trans, ms; optimized=true)
    apriori_tid(trans, ms; optimized=false)
    
    t_opt = @elapsed apriori_tid(trans, ms; optimized=true)
    t_basic = @elapsed apriori_tid(trans, ms; optimized=false)
    
    push!(opt_results, (name, ms, t_basic*1000, t_opt*1000))
    @printf("  %s (minsup=%d): Basic=%.1f ms, Optimized=%.1f ms, Speedup=%.2fx\n",
        name, ms, t_basic*1000, t_opt*1000, t_basic/t_opt)
end

In [ ]:
# Bar chart: runtime comparison
if !isempty(opt_results)
    names = [r[1] for r in opt_results]
    t_basic = [r[3] for r in opt_results]
    t_opt = [r[4] for r in opt_results]
    
    p = groupedbar([t_basic t_opt],
        bar_position=:dodge,
        bar_width=0.35,
        xticks=(1:length(names), names),
        xlabel="Dataset",
        ylabel="Thời gian (ms)",
        title="So sánh tốc độ: Basic vs Optimized",
        label=["Basic" "Optimized"],
        legend=:topright
    )
    display(p)
    savefig(p, joinpath(@__DIR__, "..", "docs", "optimization_comparison.pdf"))
end

## Tổng kết

Các charts đã được lưu dưới dạng PDF tại thư mục `docs/`. Để sử dụng trong báo cáo LaTeX:

```latex
\includegraphics[width=\textwidth]{../docs/runtime_chess.pdf}
```